# Exercises XP : Evaluating LLMs for Summarization

Cadre complet pour évaluer des LLMs sur des tâches de synthèse :
- **Partie I** : Installation
- **Partie II** : Chargement du dataset
- **Partie III** : Résumé avec T5
- **Partie IV** : Évaluation par précision exacte
- **Partie V** : Métrique ROUGE
- **Partie VI** : Analyse des scores ROUGE
- **Partie VII** : Comparaison t5-small / t5-base / gpt2
- **Partie VIII** : Agrégation et comparaison finale

---
## Partie I. Configuration

In [ ]:
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

print("✅ Installation terminée.")

---
## Partie II. Chargement et exploration du dataset

On charge `abisee/cnn_dailymail` depuis Hugging Face. Si le téléchargement échoue, un petit dataset de secours est utilisé.

In [ ]:
import pandas as pd
from datasets import load_dataset

# Chemins locaux optionnels (laisser vide pour utiliser HF)
train_path = ''
test_path  = ''

# Dataset de secours minimal
fallback = pd.DataFrame([
    {'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
     'prompt_title': 'Cat rests on mat at sunset'},
    {'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
     'prompt_title': 'Water found on the moon'},
    {'prompt_text': 'The local team won the championship after a dramatic final match.',
     'prompt_title': 'Local team clinches title'},
])

def load_and_sample(path, split_name, n):
    """Charge depuis CSV, HF Hub, ou fallback. Retourne n exemples."""
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(
                columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df  = load_and_sample(test_path,  'test',   50)

print(f"Train : {len(train_df)} exemples | Test : {len(test_df)} exemples")
print("\nPremier exemple d'entraînement :")
print("Article :", train_df['prompt_text'].iloc[0][:300], "...")
print("Résumé  :", train_df['prompt_title'].iloc[0])

print("\n--- Train DataFrame ---")
display(train_df.head(3))
print("\n--- Test DataFrame ---")
display(test_df.head(3))

---
## Partie III. Résumé avec T5

- `batch_generator` : découpe la liste d'articles en mini-lots
- `summarize_with_t5` : tokenise avec le préfixe `"summarize: "`, génère et décode

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import List

def batch_generator(items: List[str], batch_size: int):
    """Génère des sous-listes (batches) de taille batch_size."""
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


def summarize_with_t5(
    texts: List[str],
    model_name: str = 't5-small',
    batch_size: int = 4,
    max_new_tokens: int = 64
) -> List[str]:
    """
    Génère des résumés avec un modèle T5.

    Étapes :
    1. Charger tokenizer et modèle, l'envoyer sur GPU si disponible
    2. Pour chaque batch : préfixer 'summarize: ', tokeniser, générer, décoder
    3. Vider le cache CUDA entre chaque batch et en fin de fonction
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  [{model_name}] Chargement sur {device}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    model.eval()

    all_summaries = []

    for batch_idx, batch in enumerate(batch_generator(texts, batch_size)):
        # Préfixe obligatoire pour T5
        prefixed = ["summarize: " + t for t in batch]

        inputs = tokenizer(
            prefixed,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        all_summaries.extend(decoded)

        # Libérer la mémoire GPU entre chaque batch
        del inputs, output_ids
        torch.cuda.empty_cache()
        gc.collect()

        if (batch_idx + 1) % 5 == 0:
            print(f"    Batch {batch_idx+1} traité...")

    # Nettoyage final
    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"  [{model_name}] ✅ {len(all_summaries)} résumés générés.")
    return all_summaries


# Mettre RUN_T5 = True pour lancer la génération (peut prendre ~5-10 min)
RUN_T5 = False

if RUN_T5:
    train_summaries_t5 = summarize_with_t5(
        train_df['prompt_text'].tolist(),
        model_name='t5-small',
        batch_size=2
    )
    display(pd.DataFrame({
        'prompt_text':       train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary':  train_summaries_t5
    }).head())
else:
    print("⏭️  Génération T5 ignorée. Mettre RUN_T5=True pour l'exécuter.")

---
## Partie IV. Évaluation par précision exacte (exact-match accuracy)

**Pourquoi la précision est-elle presque toujours 0 pour la synthèse ?**  
La précision exacte exige que la prédiction soit *identique* mot pour mot au résumé de référence. Or, pour un même article, des dizaines de résumés valides existent — le modèle peut produire une paraphrase correcte et malgré tout obtenir un score de 0. C'est une métrique adaptée à la classification, pas à la génération de texte libre.

In [ ]:
from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    """Précision par correspondance exacte (insensible aux espaces en début/fin)."""
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)


if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy (t5-small) : {acc:.4f}")
    print("→ Ce score est quasi nul car aucune prédiction n'est mot-pour-mot identique à la référence.")
else:
    # Démonstration sur un exemple synthétique
    demo_preds = ["Cat rests on mat at sunset", "Water found on the moon", "wrong answer"]
    demo_refs  = ["Cat rests on mat at sunset", "Water found on the moon", "Local team clinches title"]
    acc_demo = compute_accuracy(demo_preds, demo_refs)
    print(f"Démonstration accuracy (2/3 corrects) : {acc_demo:.4f}  (attendu ≈ 0.667)")
    print("\nPrécision ignorée (pas de prédictions T5). Mettre RUN_T5=True.")

---
## Partie V. Implémentation de la métrique ROUGE

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) mesure le chevauchement de n-grammes entre la prédiction et la référence :  
- `ROUGE-1` : unigrammes  
- `ROUGE-2` : bigrammes  
- `ROUGE-L` : plus longue sous-séquence commune

On normalise en séparant les phrases par des sauts de ligne pour un meilleur ROUGE-L.

In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text: str) -> str:
    """Sépare les phrases par des sauts de ligne pour un meilleur ROUGE-L."""
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)  # join avec \n (pas "" comme dans le stub original)


def compute_rouge_score(preds: List[str], refs: List[str]) -> dict:
    """
    Calcule les scores ROUGE-1, ROUGE-2, ROUGE-L et ROUGE-Lsum.

    - Normalise chaque texte (séparation des phrases par \n)
    - Appelle rouge.compute avec use_stemmer=True pour robustesse aux flexions
    - Retourne un dict avec les scores moyens
    """
    norm_preds = [normalize_text(p) if p.strip() else " " for p in preds]
    norm_refs  = [normalize_text(r) for r in refs]

    scores = rouge.compute(
        predictions=norm_preds,
        references=norm_refs,
        use_stemmer=True
    )
    return scores


# Smoke test
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]

scores = compute_rouge_score(test_preds, test_refs)
print("ROUGE sanity check :")
for k, v in scores.items():
    print(f"  {k}: {v:.4f}")

---
## Partie VI. Comprendre les scores ROUGE

Expériences pour développer l'intuition sur ROUGE.

In [ ]:
# ── 1. Correspondance exacte ─────────────────────────────────────────────────
preds_exact = ["The cat sat on the mat"]
refs_exact  = ["The cat sat on the mat"]

s = compute_rouge_score(preds_exact, refs_exact)
print("1. Correspondance exacte :")
for k, v in s.items(): print(f"   {k}: {v:.4f}")
print("   → Tous les scores = 1.0 (correspondance parfaite)\n")

# ── 2. Prédiction vide ───────────────────────────────────────────────────────
preds_empty = [""]
refs_empty  = ["The cat sat on the mat"]

s = compute_rouge_score(preds_empty, refs_empty)
print("2. Prédiction vide :")
for k, v in s.items(): print(f"   {k}: {v:.4f}")
print("   → Tous les scores = 0.0 (aucun chevauchement)\n")

In [ ]:
# ── 3. Effet de la racinisation (stemming) ───────────────────────────────────
# Sans stemmer, "running" ≠ "run" → score plus bas
# Avec stemmer, les deux sont réduits à la même racine → score plus haut

pred_stem = ["The dog is running in the park"]
ref_stem  = ["The dog runs in the park"]

s_no_stem  = rouge.compute(predictions=pred_stem, references=ref_stem, use_stemmer=False)
s_yes_stem = rouge.compute(predictions=pred_stem, references=ref_stem, use_stemmer=True)

print("3. Effet du stemming — 'running' vs 'runs' :")
print(f"   ROUGE-1 sans stemmer : {s_no_stem['rouge1']:.4f}")
print(f"   ROUGE-1 avec stemmer : {s_yes_stem['rouge1']:.4f}")
print("   → Le stemmer augmente le score car 'running' et 'runs' ont la même racine.\n")

In [ ]:
# ── 4. Analyse N-gram : ROUGE-1 vs ROUGE-2 ──────────────────────────────────
examples = [
    ("dog cat bird",    "dog cat fish",   "Chevauchement partiel"),
    ("the quick brown", "the quick fox",  "Bigramme 'the quick' commun"),
    ("hello world",     "goodbye world",  "Un seul mot commun"),
    ("completely different text", "nothing is shared here", "Aucun chevauchement"),
]

print("4. ROUGE-1 vs ROUGE-2 selon le chevauchement :")
for pred, ref, label in examples:
    s = rouge.compute(predictions=[pred], references=[ref], use_stemmer=True)
    print(f"   [{label}]")
    print(f"     pred='{pred}' | ref='{ref}'")
    print(f"     ROUGE-1={s['rouge1']:.3f}  ROUGE-2={s['rouge2']:.3f}\n")

print("   → ROUGE-2 est plus strict : un bigramme entier doit correspondre.")
print("     Il diminue plus vite que ROUGE-1 quand le chevauchement est partiel.")

In [ ]:
# ── 5. Symétrie : swapper preds et refs ─────────────────────────────────────
pred_sym = ["The cat sat on the mat"]
ref_sym  = ["The mat had a cat sitting"]

s_normal  = rouge.compute(predictions=pred_sym, references=ref_sym,  use_stemmer=True)
s_swapped = rouge.compute(predictions=ref_sym,  references=pred_sym, use_stemmer=True)

print("5. Symétrie (swap preds/refs) :")
print(f"   Normal  → ROUGE-1={s_normal['rouge1']:.4f}  ROUGE-L={s_normal['rougeL']:.4f}")
print(f"   Swappé  → ROUGE-1={s_swapped['rouge1']:.4f}  ROUGE-L={s_swapped['rougeL']:.4f}")
print("   → ROUGE-1 est symétrique (F1 = harmonic mean de P et R).")
print("     ROUGE-L peut légèrement différer car la LCS est orientée ordre.")

---
## Partie VII. Comparaison t5-small / t5-base / gpt2

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as CausalTokenizer

def summarize_with_gpt2(
    texts: List[str],
    model_name: str = 'gpt2',
    batch_size: int = 2,
    max_new_tokens: int = 32
) -> List[str]:
    """
    Génère des résumés style TL;DR avec GPT-2.

    GPT-2 est un modèle génératif causal (pas seq2seq) :
    - On concatène l'article + 'TL;DR:' pour guider la génération
    - On tronque l'entrée pour rester sous la limite de 1024 tokens
    - On extrait uniquement la partie générée après 'TL;DR:'
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  [gpt2] Chargement sur {device}...")

    tokenizer = CausalTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 n'a pas de pad token natif
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()

    all_summaries = []

    for batch in batch_generator(texts, batch_size):
        # Construire les prompts TL;DR
        # Tronquer l'article pour laisser de la place à la génération
        prompts = []
        for text in batch:
            words = text.split()[:200]  # ~200 mots max pour éviter de dépasser 1024 tokens
            prompts.append(" ".join(words) + "\n\nTL;DR:")

        inputs = tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=900  # Garde 124 tokens pour la génération
        ).to(device)

        input_length = inputs['input_ids'].shape[1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Extraire uniquement les tokens générés (après le prompt)
        new_tokens = output_ids[:, input_length:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        # Nettoyer : garder uniquement la première ligne
        cleaned = [d.split('\n')[0].strip() for d in decoded]
        all_summaries.extend(cleaned)

        del inputs, output_ids, new_tokens
        torch.cuda.empty_cache()
        gc.collect()

    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"  [gpt2] ✅ {len(all_summaries)} résumés générés.")
    return all_summaries


def compute_rouge_per_row(
    df: pd.DataFrame,
    pred_col: str,
    ref_col: str = 'prompt_title'
) -> pd.DataFrame:
    """
    Calcule ROUGE-1, ROUGE-2 et ROUGE-L pour chaque ligne individuellement.
    Retourne le DataFrame enrichi de 3 nouvelles colonnes.
    """
    r1_scores, r2_scores, rl_scores = [], [], []

    for _, row in df.iterrows():
        pred = str(row[pred_col]) if pd.notna(row[pred_col]) else ""
        ref  = str(row[ref_col])
        s = rouge.compute(
            predictions=[normalize_text(pred) if pred.strip() else " "],
            references=[normalize_text(ref)],
            use_stemmer=True
        )
        r1_scores.append(round(s['rouge1'], 4))
        r2_scores.append(round(s['rouge2'], 4))
        rl_scores.append(round(s['rougeL'], 4))

    result_df = df.copy()
    result_df[f'{pred_col}_rouge1'] = r1_scores
    result_df[f'{pred_col}_rouge2'] = r2_scores
    result_df[f'{pred_col}_rougeL'] = rl_scores
    return result_df


print("✅ summarize_with_gpt2 et compute_rouge_per_row définis.")

In [ ]:
# Mettre RUN_COMPARE = True pour lancer la comparaison des 3 modèles
# ⚠️  Peut prendre 20-40 min sans GPU
RUN_COMPARE = False

if RUN_COMPARE:
    # Sous-ensemble réduit pour limiter le temps de calcul
    subset = train_df.head(10).reset_index(drop=True)
    texts  = subset['prompt_text'].tolist()

    print("Génération avec t5-small...")
    subset['t5_small']  = summarize_with_t5(texts, 't5-small',  batch_size=2)

    print("Génération avec t5-base...")
    subset['t5_base']   = summarize_with_t5(texts, 't5-base',   batch_size=2)

    print("Génération avec gpt2...")
    subset['gpt2_summ'] = summarize_with_gpt2(texts, 'gpt2',    batch_size=2)

    # Scores ROUGE par ligne pour chaque modèle
    for col in ['t5_small', 't5_base', 'gpt2_summ']:
        subset = compute_rouge_per_row(subset, pred_col=col)

    print("\nScores ROUGE par ligne (t5-small) :")
    display(subset[['prompt_title', 't5_small', 't5_small_rouge1', 't5_small_rouge2', 't5_small_rougeL']].head())

    print("\nScores ROUGE par ligne (t5-base) :")
    display(subset[['prompt_title', 't5_base',  't5_base_rouge1',  't5_base_rouge2',  't5_base_rougeL']].head())

    print("\nScores ROUGE par ligne (gpt2) :")
    display(subset[['prompt_title', 'gpt2_summ', 'gpt2_summ_rouge1', 'gpt2_summ_rouge2', 'gpt2_summ_rougeL']].head())

    # Sauvegarder pour la Partie VIII
    compare_df = subset.copy()
else:
    print("⏭️  Comparaison ignorée. Mettre RUN_COMPARE=True pour l'exécuter.")

---
## Partie VIII. Comparaison de tous les modèles

In [ ]:
def compare_models(rouge_dict: dict) -> pd.DataFrame:
    """
    Agrège les scores ROUGE moyens de plusieurs modèles.

    Paramètre :
        rouge_dict : {nom_modele: {rouge1: float, rouge2: float, rougeL: float}}

    Retourne un DataFrame avec une ligne par modèle.
    """
    rows = []
    for model_name, scores in rouge_dict.items():
        rows.append({
            'model':   model_name,
            'rouge1':  round(scores.get('rouge1', 0), 4),
            'rouge2':  round(scores.get('rouge2', 0), 4),
            'rougeL':  round(scores.get('rougeL', 0), 4),
        })
    return pd.DataFrame(rows).set_index('model').sort_values('rougeL', ascending=False)


def compare_models_summaries(df: pd.DataFrame, pred_cols: list) -> pd.DataFrame:
    """
    Affiche les résumés de chaque modèle côte à côte.

    Paramètres :
        df        : DataFrame contenant les colonnes de résumés
        pred_cols : liste des noms de colonnes de prédictions

    Retourne un DataFrame avec l'article, la référence et tous les résumés.
    """
    cols = ['prompt_text', 'prompt_title'] + [c for c in pred_cols if c in df.columns]
    return df[cols]


# ── Démonstration avec des scores fictifs (si RUN_COMPARE=False) ──────────────
if 'compare_df' in locals():
    # Calculer les scores moyens réels depuis compare_df
    rouge_results = {}
    for model_col, name in [('t5_small', 't5-small'), ('t5_base', 't5-base'), ('gpt2_summ', 'gpt2')]:
        if f'{model_col}_rouge1' in compare_df.columns:
            rouge_results[name] = {
                'rouge1': compare_df[f'{model_col}_rouge1'].mean(),
                'rouge2': compare_df[f'{model_col}_rouge2'].mean(),
                'rougeL': compare_df[f'{model_col}_rougeL'].mean(),
            }

    print("📊 Scores ROUGE agrégés (moyennes) :")
    display(compare_models(rouge_results))

    print("\n📝 Comparaison des résumés côte à côte :")
    display(compare_models_summaries(compare_df, ['t5_small', 't5_base', 'gpt2_summ']).head(5))

else:
    # Démonstration avec scores fictifs
    demo_rouge = {
        't5-small': {'rouge1': 0.312, 'rouge2': 0.118, 'rougeL': 0.274},
        't5-base':  {'rouge1': 0.351, 'rouge2': 0.147, 'rougeL': 0.309},
        'gpt2':     {'rouge1': 0.198, 'rouge2': 0.062, 'rougeL': 0.172},
    }
    print("📊 Démonstration — Scores ROUGE moyens (valeurs fictives) :")
    display(compare_models(demo_rouge))
    print("\nMettre RUN_COMPARE=True pour obtenir les vrais scores.")

---
## Wrap-up : Réflexion analytique

### Quelles métriques sont les plus informatives ?
**ROUGE-L** est la plus informative pour la synthèse : elle capture l'ordre des mots via la plus longue sous-séquence commune, ce que ROUGE-1 et ROUGE-2 ignorent. La précision exacte (*exact-match*) est inadaptée — un résumé sémantiquement parfait mais paraphrasé obtient un score de 0.

### Impact de la taille du modèle
En général : `t5-base` > `t5-small` > `gpt2` sur les scores ROUGE pour la synthèse. GPT-2 n'a pas été entraîné à résumer (il est génératif et tronque souvent le texte sans synthétiser). T5 a été pré-entraîné spécifiquement sur des tâches de synthèse avec le préfixe `"summarize: "`.

### Où la précision exacte échoue-t-elle ?
Dès qu'il y a la moindre paraphrase, synonyme, ou ordre de mots différent — ce qui est systématiquement le cas en génération de texte libre. La précision exacte est adaptée à la **classification** (étiquette unique et discrète), pas à la **génération** (espace de sorties quasi infini).

### Extensions possibles
- **Évaluation humaine** : demander à des annotateurs de noter la pertinence et la cohérence (1-5)
- **BERTScore** : similarité sémantique via embeddings contextuels, plus robuste que ROUGE aux paraphrases
- **Probes adversariaux** : tester la robustesse avec des articles bruités ou des questions hors-domaine
- **Hallucination detection** : vérifier si les résumés contiennent des faits absents de l'article source